# Notebook 3: Building a Sequence Similarity Network

**Chem 274A · Lab Week 11 · Sequence Similarity Networks**

<div class="alert alert-block alert-info">
<h3>Learning goals</h3>

- Compare every pair of sequences (an all-vs-all comparison) and think about how that scales.
- Build a NetworkX graph from the pairwise scores.
- Choose a similarity cutoff, find clusters, and use the network to identify unknown proteins.
- Use 3D structure to investigate a protein that sequence alone can't place.
</div>

In [ ]:
import itertools
import time

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from Bio import SeqIO, Align
from Bio.Align import substitution_matrices

from ssn_plot import draw_ssn   # small plotting helper that ships with this lab

records = list(SeqIO.parse("data/globins.fasta", "fasta"))
meta = pd.read_csv("data/globins_metadata.csv").set_index("id")

aligner = Align.PairwiseAligner(
    mode="local",
    substitution_matrix=substitution_matrices.load("BLOSUM62"),
    open_gap_score=-11,
    extend_gap_score=-1,
)
print(len(records), "sequences loaded")

Here's a version of `normalized_score` from Notebook 2, slightly sped up. Each sequence's self-score
gets computed once and saved in a dictionary, instead of being recomputed for every pair.

In [ ]:
self_scores = {record.id: aligner.score(record.seq, record.seq) for record in records}

def normalized_score(rec_a, rec_b):
    """Alignment score of two SeqRecords, scaled so identical sequences score 1.0."""
    best_possible = min(self_scores[rec_a.id], self_scores[rec_b.id])
    return aligner.score(rec_a.seq, rec_b.seq) / best_possible

normalized_score(records[0], records[1])

## Step 1: Compare every pair

To build the network we need a score for **every pair** of sequences. `itertools.combinations` gives us each
pair exactly once (no `(A, A)` and no `(B, A)` after `(A, B)`):

In [ ]:
for pair in itertools.combinations(["A", "B", "C", "D"], 2):
    print(pair)

### Scoring every pair

With *n* sequences there are *n*(*n* − 1)/2 pairs, so our 74 sequences give 74 × 73 / 2 = **2,701** pairs.
The loop below scores every pair, stores each result as a tuple `(id_a, id_b, score)` in the list `pair_scores`,
and times itself with `time.perf_counter()`.

In [ ]:
start = time.perf_counter()

pair_scores = []
for rec_a, rec_b in itertools.combinations(records, 2):
    score = normalized_score(rec_a, rec_b)
    pair_scores.append((rec_a.id, rec_b.id, score))

elapsed = time.perf_counter() - start
print(f"{len(pair_scores)} pairs in {elapsed:.2f} seconds")
pair_scores[:3]   # the first few results

### How does this scale?

Real SSNs often have **10,000 or more** sequences. The number of pairs grows like *n*², not *n*, so the next cell
uses your timing to estimate how long 10,000 sequences would take.

This is why real tools use much faster approximate searches, such as **BLAST**, **DIAMOND** and **MMseqs2**,
and run on computing clusters.

In [ ]:
time_per_pair = elapsed / len(pair_scores)
n = 10_000
n_pairs = n * (n - 1) // 2
print(f"{n_pairs:,} pairs -> about {time_per_pair * n_pairs / 3600:.1f} hours")

## Step 2: Build the network

Now we turn the scores into a NetworkX graph:

- **nodes** = sequence IDs, with attributes `protein_type` and `organism` from the metadata,
- **edges** = pairs whose score is **at or above a cutoff** (`threshold`), with the score stored as the edge `weight`.

Every sequence should be a node *even if it has no edges*, because a protein with no similar neighbors is an interesting result too.

The metadata for any ID is in `meta.loc[some_id]`, for example:

In [ ]:
meta.loc["sp|P69905|HBA_HUMAN"]

<div class="alert alert-block alert-success">
<h3>Exercise: Write <code>build_ssn</code></h3>

Complete `build_ssn(pair_scores, threshold)` below so it returns an `nx.Graph` built as described above.

*Hints:* `G.add_node(name, protein_type=..., organism=...)` and `G.add_edge(a, b, weight=...)`.
</div>

In [ ]:
def build_ssn(pair_scores, threshold):
    G = nx.Graph()
    # 1. Add every sequence as a node, with its protein_type and organism attributes
    # 2. Add an edge for every pair with score >= threshold
    return G

In [ ]:
G = build_ssn(pair_scores, threshold=0.3)
print(G.number_of_nodes(), "nodes")   # should be 74
print(G.number_of_edges(), "edges")
draw_ssn(G, title="Globin SSN, normalized score ≥ 0.3");

## Step 3: Choosing a cutoff

The cutoff is the most important choice when making an SSN, and **there is no single correct answer**.

- Too low: everything connects into one big hairball.
- Too high: the network shatters into tiny pieces.

Scientists usually try several cutoffs and pick one where the clusters line up with something they know
(like known protein functions).

<div class="alert alert-block alert-warning">
<h3>Different cutoffs</h3>

Run the cell below to draw the network at cutoffs of **0.2, 0.3, 0.5 and 0.7**.

What happens to the clusters as the cutoff goes up? Which two protein types are the last to separate?
</div>


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, threshold in zip(axes.flat, [0.2, 0.3, 0.5, 0.7]):
    draw_ssn(build_ssn(pair_scores, threshold), title=f"cutoff {threshold}", ax=ax)
plt.tight_layout()

NetworkX can find the clusters for us: `nx.connected_components(G)` gives one **set** of node names for each
group of connected nodes.

In [ ]:
G = build_ssn(pair_scores, threshold=0.3)
clusters = list(nx.connected_components(G))
print(len(clusters), "clusters")
print(sorted(len(c) for c in clusters))

<div class="alert alert-block alert-success">
<h3>Exercise: Which proteins are in each cluster?</h3>

Write a function `describe_clusters(G)` that prints, for each connected component (largest first),
its size and how many of each `protein_type` it contains.

*Hint:* `collections.Counter` counts things in a list, and `G.nodes[name]["protein_type"]` gets a node's type.

Example output:
```
size 30: Counter({'hemoglobin_alpha': 14, 'hemoglobin_beta': 14, 'unknown': 2})
size 15: Counter({'neuroglobin': 15})
...
```
</div>

In [ ]:
from collections import Counter

def describe_clusters(G):
    # Your code here
    pass

describe_clusters(build_ssn(pair_scores, threshold=0.3))

<div class="alert alert-block alert-warning">
<h3>Choose a cutoff</h3>

Now explore on your own. Change `cutoff` in the cell below and re-run it as many times as you like
(anything from 0.1 to 0.8 is worth a look). The cell draws the network and runs your `describe_clusters`.

You're looking for a cutoff where each cluster is **one color and shape** (one protein type), without lots of
proteins dropping into the row of singletons at the bottom.

1. At what cutoff do hemoglobin α and β finally split into separate clusters?
2. At what cutoff does a single protein type start breaking into more than one cluster?
3. Which cutoff do you choose? Be ready to defend it in the report-out.
</div>

In [ ]:
cutoff = 0.4   # change this and re-run

G = build_ssn(pair_scores, cutoff)
draw_ssn(G, title=f"Globin SSN, normalized score ≥ {cutoff}")
describe_clusters(G)

## Step 4: Solve the mysteries

The real power of an SSN: if an unknown protein lands in a cluster of well-studied proteins, it probably shares their function.

<div class="alert alert-block alert-warning">
<h3>What are the mystery proteins?</h3>

The mystery proteins are the black **stars**, labeled `mystery_1`, `mystery_2` and `mystery_3`. Find them in the network
at your chosen cutoff, and in the four panels at the start of Step 3.

For each mystery protein:
1. Which cluster is it in? What type of protein are its neighbors?
2. Does that change as the cutoff goes up or down? (Compare 0.3 with 0.5 for `mystery_1` and `mystery_2`.)
3. Make a guess: what kind of protein is it?

Write your answers in the table below (double-click it to edit). One mystery protein won't fit neatly anywhere.
We'll follow that up in Step 5.
</div>

| | Cluster at my cutoff | What changes with the cutoff | My guess |
|---|---|---|---|
| `mystery_1` | | | |
| `mystery_2` | | | |
| `mystery_3` | | | |

## Step 5: A second opinion from 3D structure

An SSN only sees sequence, but **folds change much more slowly than sequences do**. Two proteins can drift apart until
their sequences barely match while their structures stay nearly the same. So when a protein sits alone in an SSN,
the next question a protein scientist asks today is: *what does its structure look like?*

`data/structures/` holds crystal structures from the [Protein Data Bank](https://www.rcsb.org): `mystery_3`, four
kinds of globin, and two **controls**, which are proteins of similar size that are mostly α-helical but are *not* globins.
The myoglobin and neuroglobin crystals carry a few engineered point mutations, which are too small to change the overall fold.
If no crystal structure exists, a predicted one from the [AlphaFold Database](https://alphafold.ebi.ac.uk)
is often good enough for this kind of comparison.

We'll compare structures with **TM-align**, which finds the best 3D superposition of two chains and reports a
**TM-score**. Like your normalized score, it's scaled for length so that scores are comparable between proteins:

| TM-score | Meaning |
|---|---|
| 1.0 | identical structures |
| above 0.5 | generally the same fold |
| around 0.3 or below | about what unrelated proteins get |

The cell below loads each file with Biopython's `PDB` module and keeps the position of each α-carbon, one per residue.


In [ ]:
from pathlib import Path

import numpy as np
import tmtools
from Bio.Data.PDBData import protein_letters_3to1
from Bio.PDB import PDBParser

def load_structure(path):
    """Return the α-carbon coordinates (n × 3 array) and sequence of the chain in a PDB file."""
    chain = next(PDBParser(QUIET=True).get_structure(path.stem, path).get_chains())
    residues = [residue for residue in chain if "CA" in residue]
    coords = np.array([residue["CA"].coord for residue in residues], dtype=float)
    seq = "".join(protein_letters_3to1.get(residue.get_resname(), "X") for residue in residues)
    return coords, seq

structures = {path.stem: load_structure(path) for path in sorted(Path("data/structures").glob("*.pdb"))}
for name, (coords, seq) in structures.items():
    print(f"{name:26s} {len(seq)} residues")

<div class="alert alert-block alert-warning">
<h3>Compare the structures</h3>

The next cell computes the TM-score between `mystery_3` and every other structure and prints them from highest to
lowest. Then it prints the sequence clusters again at a looser cutoff of 0.2.

1. Which structures score above 0.5? Where do the two controls land, and why did we include them?
2. In Step 4, `mystery_3` stayed on its own except at very low cutoffs. What does the structure tell you that the
   sequence didn't?
3. At a sequence cutoff of 0.2, where does `mystery_3` end up? Would that alone have convinced you?
</div>

In [ ]:
mystery_coords, mystery_seq = structures["mystery_3"]

tm_scores = {}
for name, (coords, seq) in structures.items():
    if name != "mystery_3":
        result = tmtools.tm_align(mystery_coords, coords, mystery_seq, seq)
        tm_scores[name] = result.tm_norm_chain1

for name in sorted(tm_scores, key=tm_scores.get, reverse=True):
    print(f"{name:26s} {tm_scores[name]:.2f}")

print("\nClusters at a sequence cutoff of 0.2:")
describe_clusters(build_ssn(pair_scores, threshold=0.2))

## Step 6: Export for Cytoscape (optional)

Scientists usually explore large SSNs in [Cytoscape](https://cytoscape.org), a free program for viewing networks.
EFI-EST gives you a Cytoscape file. With NetworkX we can write one ourselves in GraphML format,
which Cytoscape opens directly (*File → Import → Network from File*).

In [ ]:
G = build_ssn(pair_scores, threshold=cutoff)   # the cutoff you chose in Step 3
nx.write_graphml(G, "globin_ssn.graphml")
print("wrote globin_ssn.graphml")

## Summary

You've just built a sequence similarity network from scratch:

1. read sequences (Biopython `SeqIO`),
2. scored every pair (Biopython `PairwiseAligner` and `itertools.combinations`),
3. built and thresholded a graph (NetworkX),
4. used clusters to classify unknown proteins,
5. used 3D structure (TM-align) to place the protein that sequence couldn't.

That's the same pipeline as the EFI-EST web tool, minus the speed.

**Next:** discuss the Report Out questions in the README with your group. If you have time, try **Notebook 4**
to run MMseqs2 and Foldseek, two fast search tools used in current research.
